# VAE: C 1s peak properties

A latent representation supports both spectrum reconstruction and peak-property prediction. Each model predicts peak presence, center, measured FWHM, area, and shape for five C 1s components.

## 1. Load SESSA spectra and labels

Set `data_path` to the generated NPZ on the training machine. Derive each present peak's FWHM from its saved SESSA component curve. Keep all noisy versions of a case in the same split.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = Path("datasets/sessa_c1s_v1/dataset.npz")  # Change on the training machine if needed.
assert data_path.exists(), f"Set data_path to the SESSA dataset on your training machine: {data_path}"
data = np.load(data_path)
energy = data["energy_ev"].astype(np.float32)
names = data["components"].tolist()
base_centers = {"CC": 285.0, "CO": 286.5, "CARBONYL": 287.7,
                "OCO": 288.1, "COO": 289.1}
base = np.array([base_centers[name] for name in names], dtype=np.float32)
base_tensor = torch.tensor(base, device=device)
shape_names = ["Gaussian", "Lorentzian", "pseudo-Voigt"]

def prepare_spectrum(values):
    values = np.asarray(values, dtype=np.float32)
    left = values[..., :10].mean(axis=-1, keepdims=True)
    right = values[..., -10:].mean(axis=-1, keepdims=True)
    line = left + (right - left) * np.linspace(0, 1, values.shape[-1], dtype=np.float32)
    corrected = np.maximum(values - line, 0)
    return corrected / np.maximum(corrected.max(axis=-1, keepdims=True), 1e-6)

def measured_fwhm(curve):
    """Width at half maximum of a saved SESSA component curve."""
    top = int(np.argmax(curve))
    half = curve[top] / 2
    left = np.flatnonzero(curve[:top] < half)
    right = np.flatnonzero(curve[top + 1:] < half)
    if not len(left) or not len(right):
        raise ValueError("A component has no measurable FWHM on the energy grid")
    i = left[-1]
    j = top + 1 + right[0]
    x_left = np.interp(half, [curve[i], curve[i + 1]], [energy[i], energy[i + 1]])
    x_right = np.interp(half, [curve[j], curve[j - 1]], [energy[j], energy[j - 1]])
    return x_right - x_left

presence = data["presence"].astype(bool)
component_curves = data["component_relative"]
width = np.zeros(presence.shape, dtype=np.float32)
for case, component in np.argwhere(presence):
    width[case, component] = measured_fwhm(component_curves[case, component])

# Label columns: present, center (eV), FWHM (eV), area (relative intensity × eV), shape code.
labels = np.zeros((*presence.shape, 5), dtype=np.float32)
labels[..., 0] = presence
labels[..., 1] = np.where(presence, data["center_ev"], base[None, :])
labels[..., 2] = width
labels[..., 3] = data["component_area_window"]
labels[..., 4] = np.where(presence, data["shape_code"], 0)

noise_levels = data["peak_counts"]
n_noise = len(noise_levels)
spectra = prepare_spectrum(data["poisson_counts"]).reshape(-1, len(energy))
targets = np.repeat(labels, n_noise, axis=0)
splits = np.repeat(data["split"], n_noise)
case_indices = np.repeat(np.arange(len(data["case_id"])), n_noise)
noise_indices = np.tile(np.arange(n_noise), len(data["case_id"]))

def make_loader(split, shuffle=False):
    chosen = splits == split
    dataset = TensorDataset(torch.from_numpy(spectra[chosen]),
                            torch.from_numpy(targets[chosen]),
                            torch.from_numpy(case_indices[chosen]),
                            torch.from_numpy(noise_indices[chosen]))
    return DataLoader(dataset, batch_size=128, shuffle=shuffle)

train_loader = make_loader("train", shuffle=True)
val_loader = make_loader("validation")
test_loader = make_loader("test")
print(f"SESSA cases: {len(data['case_id'])}; training scans: {len(train_loader.dataset)}")
print("Peak types:", names)


SESSA cases: 10000; training scans: 23913
Peak types: ['CC', 'CO', 'CARBONYL', 'OCO', 'COO']


## 2. Define the VAE

The final layer gives seven values per component: one presence score, center, FWHM, area, and three shape scores.

In [2]:
class VAE(nn.Module):
    def __init__(self, points):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(points, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU())
        self.mean = nn.Linear(64, 16)
        self.log_variance = nn.Linear(64, 16)
        self.decoder = nn.Sequential(nn.Linear(16, 128), nn.ReLU(), nn.Linear(128, points), nn.Sigmoid())
        self.head = nn.Linear(16, 5 * 7)

    def forward(self, x):
        hidden = self.encoder(x)
        mean = self.mean(hidden)
        log_variance = self.log_variance(hidden)
        z = mean + torch.randn_like(mean) * torch.exp(0.5 * log_variance) if self.training else mean
        return self.head(z), self.decoder(z), mean, log_variance

model = VAE(len(energy)).to(device)

def peak_outputs(raw):
    """Turn 35 network values into five sets of peak properties."""
    raw = raw.reshape(-1, len(names), 7)
    return {
        "presence_logit": raw[..., 0],
        "center": base_tensor + 0.6 * torch.tanh(raw[..., 1]),
        "fwhm": 0.4 + 1.2 * torch.sigmoid(raw[..., 2]),
        "area": F.softplus(raw[..., 3]),
        "shape_logit": raw[..., 4:7],
    }

print("Trainable parameters:", sum(p.numel() for p in model.parameters()))


Trainable parameters: 116292


## 3. Train

Compare every predicted property with its SESSA label. Save the model with the lowest validation loss.

In [13]:
epochs = 100
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
checkpoint = Path("outputs/sessa_models/vae.pt")
checkpoint.parent.mkdir(parents=True, exist_ok=True)
best_validation = float("inf")

def loss_on_batch(x, y):
    output = model(x)
    raw = output[0] if isinstance(output, tuple) else output
    pred = peak_outputs(raw)
    present = y[..., 0]
    mask = present > 0.5
    n_present = mask.sum().clamp(min=1)
    presence_loss = F.binary_cross_entropy_with_logits(pred["presence_logit"], present)
    center_loss = (((pred["center"] - y[..., 1]) / 0.5).square() * mask).sum() / n_present
    width_loss = (((pred["fwhm"] - y[..., 2]) / 1.0).square() * mask).sum() / n_present
    area_loss = (((pred["area"] - y[..., 3]) / 2.0).square() * mask).sum() / n_present
    shape_loss = F.cross_entropy(pred["shape_logit"][mask], y[..., 4][mask].long())
    loss = presence_loss + center_loss + width_loss + area_loss + 0.25 * shape_loss
    if isinstance(output, tuple):
        _, reconstruction, mean, log_variance = output
        reconstruction_loss = F.mse_loss(reconstruction, x)
        kl_loss = -0.5 * torch.mean(1 + log_variance - mean.square() - log_variance.exp())
        loss = loss + 0.1 * reconstruction_loss + 0.001 * kl_loss
    return loss

for epoch in range(1, epochs + 1):
    model.train()
    for x, y, _, _ in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_on_batch(x, y)
        loss.backward()
        optimizer.step()

    model.eval()
    validation_loss = 0.0
    with torch.no_grad():
        for x, y, _, _ in val_loader:
            validation_loss += loss_on_batch(x.to(device), y.to(device)).item() * len(x)
    validation_loss /= len(val_loader.dataset)
    if validation_loss < best_validation:
        best_validation = validation_loss
        torch.save(model.state_dict(), checkpoint)
    print(f"Epoch {epoch:2d}: validation loss = {validation_loss:.4f}")
print("Best model:", checkpoint)


Epoch  1: validation loss = 0.2412
Epoch  2: validation loss = 0.2393
Epoch  3: validation loss = 0.2372
Epoch  4: validation loss = 0.2391
Epoch  5: validation loss = 0.2397
Epoch  6: validation loss = 0.2390
Epoch  7: validation loss = 0.2418
Epoch  8: validation loss = 0.2393
Epoch  9: validation loss = 0.2392
Epoch 10: validation loss = 0.2373
Epoch 11: validation loss = 0.2377
Epoch 12: validation loss = 0.2422
Epoch 13: validation loss = 0.2384
Epoch 14: validation loss = 0.2380
Epoch 15: validation loss = 0.2381
Epoch 16: validation loss = 0.2382
Epoch 17: validation loss = 0.2381
Epoch 18: validation loss = 0.2371
Epoch 19: validation loss = 0.2402
Epoch 20: validation loss = 0.2396
Epoch 21: validation loss = 0.2421
Epoch 22: validation loss = 0.2370
Epoch 23: validation loss = 0.2364
Epoch 24: validation loss = 0.2373
Epoch 25: validation loss = 0.2394
Epoch 26: validation loss = 0.2360
Epoch 27: validation loss = 0.2383
Epoch 28: validation loss = 0.2350
Epoch 29: validation

## 4. Measure test accuracy

Use held-out SESSA cases. Report presence precision, recall and F1; center, FWHM and area errors; and shape accuracy.

In [14]:
model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
model.eval()
predicted, actual, test_cases, test_noise = [], [], [], []
with torch.no_grad():
    for x, y, case, noise in test_loader:
        output = model(x.to(device))
        raw = output[0] if isinstance(output, tuple) else output
        p = peak_outputs(raw)
        predicted.append(np.stack([
            torch.sigmoid(p["presence_logit"]).cpu().numpy(),
            p["center"].cpu().numpy(), p["fwhm"].cpu().numpy(),
            p["area"].cpu().numpy(),
            p["shape_logit"].argmax(dim=-1).cpu().numpy()], axis=-1))
        actual.append(y.numpy())
        test_cases.append(case.numpy())
        test_noise.append(noise.numpy())
predicted = np.concatenate(predicted)
actual = np.concatenate(actual)
test_cases = np.concatenate(test_cases)
test_noise = np.concatenate(test_noise)

def property_metrics(p, y):
    def one_row(estimate, truth, name):
        true_present = truth[:, 0] > 0.5
        pred_present = estimate[:, 0] >= 0.5
        tp = np.sum(true_present & pred_present)
        precision = tp / max(np.sum(pred_present), 1)
        recall = tp / max(np.sum(true_present), 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)
        return {
            "component": name,
            "presence_precision": precision,
            "presence_recall": recall,
            "presence_F1": f1,
            "center_MAE_eV": np.mean(np.abs(estimate[true_present, 1] - truth[true_present, 1])),
            "FWHM_MAE_eV": np.mean(np.abs(estimate[true_present, 2] - truth[true_present, 2])),
            "area_MAE": np.mean(np.abs(estimate[true_present, 3] - truth[true_present, 3])),
            "shape_accuracy": np.mean(estimate[true_present, 4] == truth[true_present, 4]),
        }
    rows = [one_row(p[:, j], y[:, j], name) for j, name in enumerate(names)]
    rows.append(one_row(p.reshape(-1, 5), y.reshape(-1, 5), "ALL"))
    return pd.DataFrame(rows).set_index("component")

metrics = property_metrics(predicted, actual)
print("Held-out SESSA test metrics (property errors are for truly present peaks):")
print(metrics.round(3).to_string())
metrics.to_csv(checkpoint.with_name("vae_test_metrics.csv"))


Held-out SESSA test metrics (property errors are for truly present peaks):
           presence_precision  presence_recall  presence_F1  center_MAE_eV  FWHM_MAE_eV  area_MAE  shape_accuracy
component                                                                                                        
CC                      0.999            0.999        0.999          0.022        0.044     0.051           0.726
CO                      0.999            0.997        0.998          0.033        0.057     0.063           0.609
CARBONYL                0.977            0.959        0.968          0.048        0.061     0.040           0.529
OCO                     0.986            0.994        0.990          0.053        0.067     0.021           0.592
COO                     0.996            0.996        0.996          0.030        0.054     0.051           0.594
ALL                     0.995            0.993        0.994          0.033        0.053     0.050           0.635


## 5. Plot hard successes and easy failures

Show the normalized noisy scan given to the model, true peak curves, and predicted peak curves. Harder scans have fewer counts and more true peaks. Success requires correct peak presence and error no worse than the test median; unsuccessful fits are the worst 10% by reconstruction error.

In [ ]:
integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def peak_profile(center, fwhm, shape):
    """Unit-height Gaussian, Lorentzian, or 50:50 pseudo-Voigt profile."""
    distance = (energy - center) / fwhm
    gaussian = np.exp(-4 * np.log(2) * distance**2)
    lorentzian = 1 / (1 + 4 * distance**2)
    mix = [0.0, 1.0, 0.5][int(shape)]
    return (1 - mix) * gaussian + mix * lorentzian

def peak_curve(center, fwhm, area, shape):
    profile = peak_profile(center, fwhm, shape)
    return profile * area / max(integrate(profile, energy), 1e-9)

def reconstruct(properties, predicted_values=False):
    total = np.zeros_like(energy, dtype=np.float64)
    for peak in properties:
        is_present = peak[0] >= 0.5 if predicted_values else peak[0] > 0.5
        if is_present:
            total += peak_curve(peak[1], peak[2], peak[3], peak[4])
    return total

scores = []
for p, y in zip(predicted, actual):
    truth = reconstruct(y)
    estimate = reconstruct(p, predicted_values=True)
    scores.append(integrate(np.abs(estimate - truth), energy) /
                  max(integrate(truth, energy), 1e-6))
scores = np.asarray(scores)
print(f"Relative reconstruction error: mean {scores.mean():.3f}, median {np.median(scores):.3f}")

# Harder inputs have fewer counts and more true components. A successful fit
# finds every peak and has at most the median reconstruction error. An
# unsuccessful fit has an error in the worst 10% of test scans.
true_peak_count = (actual[..., 0] > 0.5).sum(axis=1)
counts = np.asarray(noise_levels)[test_noise]
presence_correct = np.all((predicted[..., 0] >= 0.5) == (actual[..., 0] > 0.5), axis=1)
success_limit = np.median(scores)
failure_limit = np.quantile(scores, 0.90)
success_candidates = np.flatnonzero(presence_correct & (scores <= success_limit))
failure_candidates = np.flatnonzero(scores >= failure_limit)

def three_distinct(order, excluded=()):
    chosen, seen = [], set(excluded)
    for index in order:
        case = int(test_cases[index])
        if case not in seen:
            chosen.append(index)
            seen.add(case)
        if len(chosen) == 3:
            break
    if len(chosen) != 3:
        raise ValueError("Need three distinct cases in each success/failure group")
    return chosen

hardest_success = three_distinct(sorted(
    success_candidates,
    key=lambda i: (counts[i], -true_peak_count[i], -scores[i])))
easiest_failure = three_distinct(sorted(
    failure_candidates,
    key=lambda i: (-counts[i], true_peak_count[i], scores[i])),
    excluded={int(test_cases[i]) for i in hardest_success})
print(f"Success: all peak-presence decisions correct and error <= {success_limit:.3f} (test median).")
print(f"Unsuccessful: reconstruction error >= {failure_limit:.3f} (worst 10%).")

from matplotlib.lines import Line2D
component_colors = {"CC": "tab:blue", "CO": "tab:orange", "CARBONYL": "tab:green",
                    "OCO": "tab:purple", "COO": "tab:brown"}
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for ax, index, group in zip(axes.flat, hardest_success + easiest_failure,
                            ["Hardest successful"] * 3 + ["Easiest unsuccessful"] * 3):
    case = int(test_cases[index])
    noise = int(test_noise[index])
    observed = spectra[case * n_noise + noise]  # The normalized noisy scan fed to the model.
    for j, peak in enumerate(actual[index]):
        if peak[0] > 0.5:
            ax.plot(energy, peak_curve(peak[1], peak[2], peak[3], peak[4]),
                    color=component_colors[names[j]], lw=0.9, ls=":", alpha=0.8)
    for j, peak in enumerate(predicted[index]):
        if peak[0] >= 0.5:
            ax.plot(energy, peak_curve(peak[1], peak[2], peak[3], peak[4]),
                    color=component_colors[names[j]], lw=0.9, ls="--", alpha=0.8)
    ax.plot(energy, observed, color="black", lw=1, alpha=0.8)
    ax.plot(energy, reconstruct(actual[index]), color="tab:blue", lw=1.8)
    ax.plot(energy, reconstruct(predicted[index], True),
            color="tab:red", lw=1.8, ls="--")
    case_name = data["case_id"][case]
    ax.set_title(f"{group}: {case_name}\n{counts[index]} counts, "
                 f"{true_peak_count[index]} peaks, error {scores[index]:.3f}", fontsize=10)
    ax.set_xlabel("Binding energy (eV)")
    ax.set_ylabel("Relative intensity")
legend_lines = [
    Line2D([0], [0], color="black", lw=1, label="Noisy model input"),
    Line2D([0], [0], color="tab:blue", lw=1.8, label="True peak sum"),
    Line2D([0], [0], color="tab:red", lw=1.8, ls="--", label="Predicted peak sum"),
    Line2D([0], [0], color="gray", lw=0.9, ls=":", label="True component"),
    Line2D([0], [0], color="gray", lw=0.9, ls="--", label="Predicted component"),
]
legend_lines += [Line2D([0], [0], color=color, lw=1.3, label=component)
                 for component, color in component_colors.items()]
fig.legend(handles=legend_lines, loc="lower center", ncol=5, fontsize=8)
fig.tight_layout(rect=[0, 0.10, 1, 1])
figure_path = checkpoint.with_name("vae_best_worst.png")
fig.savefig(figure_path, dpi=150)
plt.show()
print("Saved:", figure_path)


## 6. Predict peaks on real measurements

Read the measured C 1s spectra and print the model predictions. Raw properties of peaks marked absent are shown, but are not used in the comparison or reconstructed curves.

In [16]:
real_data_path = Path("datasets/MalonicSucrose/real_c1s_spectra.csv")
assert real_data_path.exists(), f"Real spectra file missing: {real_data_path}"
real_data = pd.read_csv(real_data_path)

rows = []
model.eval()
for sample, scan in real_data.groupby("sample", sort=False):
    scan = scan.sort_values("binding_energy_ev")
    if len(scan) != len(energy) or not np.allclose(scan["binding_energy_ev"], energy):
        raise ValueError(f"Energy grid in {sample} does not match the SESSA grid")
    x = scan["model_input"].to_numpy(dtype=np.float32)
    with torch.no_grad():
        output = model(torch.from_numpy(x[None]).to(device))
        raw = output[0] if isinstance(output, tuple) else output
        p = peak_outputs(raw)
    probability = torch.sigmoid(p["presence_logit"]).cpu().numpy()[0]
    center = p["center"].cpu().numpy()[0]
    fwhm = p["fwhm"].cpu().numpy()[0]
    area = p["area"].cpu().numpy()[0]
    shape = p["shape_logit"].argmax(dim=-1).cpu().numpy()[0]
    for j, component in enumerate(names):
        rows.append({"sample": sample, "component": component,
                     "presence_probability": probability[j],
                     "present": int(probability[j] >= 0.5),
                     "center_ev": center[j], "fwhm_ev": fwhm[j],
                     "area_window": area[j], "shape": shape_names[shape[j]]})

real_peaks = pd.DataFrame(rows)
print(real_peaks.round(3).to_string(index=False))


               sample component  presence_probability  present  center_ev  fwhm_ev  area_window        shape
       Sucrose 0 mbar        CC                 1.000        1 285.424011    0.434        0.354   Lorentzian
       Sucrose 0 mbar        CO                 1.000        1 286.803009    0.406        0.656   Lorentzian
       Sucrose 0 mbar  CARBONYL                 0.000        0 288.298004    0.403        0.001   Lorentzian
       Sucrose 0 mbar       OCO                 0.001        0 288.635010    0.406        0.019 pseudo-Voigt
       Sucrose 0 mbar       COO                 0.000        0 288.509003    0.404        1.039   Lorentzian
     Sucrose 0.2 mbar        CC                 1.000        1 285.320007    0.579        0.812   Lorentzian
     Sucrose 0.2 mbar        CO                 1.000        1 286.858002    0.471        0.559   Lorentzian
     Sucrose 0.2 mbar  CARBONYL                 0.000        0 288.289001    0.477        0.007   Lorentzian
     Sucrose 0.2 mb

## 7. Compare with published fits

Compare mapped peaks with Tables S3 and S5. Absent model peaks have zero area; center and FWHM errors use detected peaks only. Published widths are divided by 1000 under the stated meV assumption. Malonic acid has only a partial component mapping.

In [ ]:
# Compare only published peaks that map clearly to a model component.
reference_path = Path("datasets/MalonicSucrose/real_c1s_comparable_fits.csv")
reference = pd.read_csv(reference_path)
comparison = reference.merge(real_peaks, left_on=["sample", "model_component"],
                             right_on=["sample", "component"], validate="one_to_one")
if len(comparison) != len(reference):
    raise ValueError("A published peak has no model prediction")

# An absent peak contributes no area, just as in the reconstructed spectrum.
comparison["detected_area"] = comparison["area_window"].where(comparison["present"] == 1, 0.0)
area_total = comparison.groupby("sample")["detected_area"].transform("sum")
comparison["model_area_fraction_of_matched_peaks"] = (
    comparison["detected_area"].div(area_total.where(area_total > 0)).fillna(0.0))

# Center spacing needs a detected CC anchor. Other properties are scored only
# where the model predicts that the peak exists.
cc = real_peaks[real_peaks["component"] == "CC"].set_index("sample")
cc_center = comparison["sample"].map(cc["center_ev"])
cc_detected = comparison["sample"].map(cc["present"]) == 1
comparison["model_center_offset_from_cc_ev"] = (
    comparison["center_ev"] - cc_center).where((comparison["present"] == 1) & cc_detected)
comparison["model_fwhm_if_present"] = comparison["fwhm_ev"].where(comparison["present"] == 1)

side_by_side = pd.DataFrame({
    "sample": comparison["sample"], "peak": comparison["model_component"],
    "P(present)": comparison["presence_probability"],
    "detected": comparison["present"],
    "published_dE_eV": comparison["published_center_offset_from_cc_ev"],
    "model_dE_eV": comparison["model_center_offset_from_cc_ev"],
    "published_FWHM_eV*": comparison["published_fwhm_ev_assuming_printed_mev"],
    "model_FWHM_eV": comparison["model_fwhm_if_present"],
    "published_area_share": comparison["published_area_fraction_of_matched_peaks"],
    "model_area_share": comparison["model_area_fraction_of_matched_peaks"],
})
print("Published fits beside model predictions (mapped peaks only):")
print(side_by_side.round(3).to_string(index=False, na_rep="n/a"))
print("* Published FWHM numbers are treated as meV; the supplement labels them eV.")
print("Absent model peaks have zero area share and no scored center or FWHM.")

def format_mae(errors, unit):
    return f"{errors.mean():.3f} {unit}" if len(errors) else "n/a"

for material, group in comparison.groupby("material", sort=False):
    detected = group[group["present"] == 1]
    non_anchor = detected[(detected["model_component"] != "CC") &
                          detected["model_center_offset_from_cc_ev"].notna()]
    center_errors = (non_anchor["model_center_offset_from_cc_ev"] -
                     non_anchor["published_center_offset_from_cc_ev"]).abs()
    width_errors = (detected["model_fwhm_if_present"] -
                    detected["published_fwhm_ev_assuming_printed_mev"]).abs()
    area_errors = (group["model_area_fraction_of_matched_peaks"] -
                   group["published_area_fraction_of_matched_peaks"]).abs()
    area_mae_pp = 100 * area_errors.mean()
    print(f"{material}: mapped peaks detected {len(detected)}/{len(group)}; "
          f"center-spacing MAE (detected) {format_mae(center_errors, 'eV')}; "
          f"assumed-width MAE (detected) {format_mae(width_errors, 'eV')}; "
          f"area-share MAE {area_mae_pp:.1f} percentage points")
sucrose_reference = reference[reference["material"] == "sucrose"]
extra_sucrose = real_peaks[
    real_peaks["sample"].isin(sucrose_reference["sample"]) &
    (real_peaks["present"] == 1) &
    ~real_peaks["component"].isin(sucrose_reference["model_component"])]
print(f"Additional detected sucrose components outside the published fit: {len(extra_sucrose)}")
print("Malonic acid scores cover CC and COO only; shape and absolute center are not scored.")


## 8. Plot predictions on the real spectra

Show the noisy normalized input, model peaks, and published fitted peaks. Published curves use an approximate pseudo-Voigt profile and are aligned to the observed C–C maximum; their exact Voigt curves are unavailable. Malonic acid shows only mapped published peaks. Each sum is scaled separately, so this compares shape rather than absolute intensity.

In [ ]:
from matplotlib.lines import Line2D

# The published table gives centers, total widths, and relative areas but not
# the exact fitted Voigt curves. Draw approximate 50:50 pseudo-Voigt curves.
reference = pd.read_csv("datasets/MalonicSucrose/real_c1s_comparable_fits.csv")
colors = {"CC": "tab:blue", "CO": "tab:orange", "CARBONYL": "tab:green",
          "OCO": "tab:purple", "COO": "tab:brown"}
shape_codes = {name: i for i, name in enumerate(shape_names)}
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
for ax, (sample, scan) in zip(axes.flat, real_data.groupby("sample", sort=False)):
    scan = scan.sort_values("binding_energy_ev")
    measured = scan["model_input"].to_numpy()  # The noisy, normalized model input.

    predicted_rows = real_peaks[(real_peaks["sample"] == sample) &
                                (real_peaks["present"] == 1)]
    predicted_curves = {}
    predicted_sum = np.zeros_like(energy, dtype=np.float64)
    for _, peak in predicted_rows.iterrows():
        curve = peak_curve(peak["center_ev"], peak["fwhm_ev"], peak["area_window"],
                           shape_codes[peak["shape"]])
        predicted_curves[peak["component"]] = curve
        predicted_sum += curve
    predicted_scale = max(float(predicted_sum.max()), 1e-6)

    published_rows = reference[reference["sample"] == sample]
    if published_rows.empty:
        raise ValueError(f"No published peak fit for {sample}")
    # Align the published CC center to the observed CC maximum. This removes
    # the reported absolute-energy offset without using the model prediction.
    cc_window = (energy >= 284.0) & (energy <= 286.5)
    smoothed = np.convolve(measured, np.ones(7) / 7, mode="same")
    observed_cc = energy[cc_window][np.argmax(smoothed[cc_window])]
    published_curves = {}
    published_sum = np.zeros_like(energy, dtype=np.float64)
    for _, peak in published_rows.iterrows():
        center = observed_cc + peak["published_center_offset_from_cc_ev"]
        curve = peak_curve(center, peak["published_fwhm_ev_assuming_printed_mev"],
                           peak["published_area_fraction_of_matched_peaks"], 2)
        published_curves[peak["model_component"]] = curve
        published_sum += curve
    published_scale = max(float(published_sum.max()), 1e-6)

    ax.plot(energy, measured, color="black", lw=1.3, label="Noisy model input")
    for component, curve in published_curves.items():
        ax.plot(energy, curve / published_scale, color=colors[component],
                lw=0.9, ls=":", alpha=0.8)
    for component, curve in predicted_curves.items():
        ax.plot(energy, curve / predicted_scale, color=colors[component],
                lw=0.9, ls="--", alpha=0.8)
    ax.plot(energy, published_sum / published_scale,
            color="tab:blue", lw=1.8, label="Published mapped sum (approx.)")
    ax.plot(energy, predicted_sum / predicted_scale,
            color="tab:red", lw=1.8, ls="--", label="Predicted sum")
    scope = " (partial published fit)" if sample.startswith("Malonic acid") else ""
    ax.set_title(sample + scope)
    ax.set_ylim(-0.03, 1.08)
for ax in axes[1]:
    ax.set_xlabel("Binding energy (eV)")
for ax in axes[:, 0]:
    ax.set_ylabel("Normalized intensity")
legend_lines = [
    Line2D([0], [0], color="black", lw=1.3, label="Noisy model input"),
    Line2D([0], [0], color="tab:blue", lw=1.8, label="Published mapped sum"),
    Line2D([0], [0], color="tab:red", lw=1.8, ls="--", label="Predicted sum"),
    Line2D([0], [0], color="gray", lw=0.9, ls=":", label="Published component"),
    Line2D([0], [0], color="gray", lw=0.9, ls="--", label="Predicted component"),
]
legend_lines += [Line2D([0], [0], color=color, lw=1.3, label=component)
                 for component, color in colors.items()]
fig.legend(handles=legend_lines, loc="lower center", ncol=5, fontsize=8)
fig.suptitle("Real C 1s model input, published fitted peaks, and predicted peaks", fontsize=13)
fig.tight_layout(rect=[0, 0.10, 1, 0.95])
figure_path = checkpoint.with_name("vae_real_data.png")
fig.savefig(figure_path, dpi=150)
plt.show()
print("Saved:", figure_path)
